In [2]:
# Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms.functional import normalize
import torchvision.models as tvmodels
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms 

import numpy as np
import rasterio

import os

from skimage.io import imread

OSError: [WinError 126] The specified module could not be found. Error loading "c:\Users\b1114674\AppData\Local\miniconda3\envs\play\lib\site-packages\torch\lib\torch_python.dll" or one of its dependencies.

In [4]:
#TOTAL NUMBER OF IMAGES ARE AROUND 21000

In [ ]:
pre_datafolder= ''
post_datafolder = ''
    
pre_pathlist = [os.path.join(pre_datafolder,img.endswith(".tif")) for img in os.listdir(pre_datafolder)]

post_pathlist= [os.path.join(post_datafolder,img.endswith(".tif")) for img in os.listdir(post_datafolder)]

In [ ]:
pre_meanstd = {"mean":[103.29,94.94, 97.53],"std":[48.63, 36.42, 31.22]}
post_meanstd = {"mean":[122.0, 110.35, 116.4],"std":[47.99, 33.89, 32.04]}

In [ ]:
# Dataset and Dataloader

class CustomDataset(Dataset):

    def __init__(self,pre,post,pre_meanstd,post_meanstd):
        super().__init__()

        self.pre_meanstd = pre_meanstd
        self.post_meanstd = post_meanstd


        self.pre = pre
        self.post = post


    def __getitem__(self, index):
        pre = self.pre[index]
        post = self.post[index]

        pre = imread(pre)
        post = imread(post)

        if pre.max()  == 65535 or post.max() == 65535:
            pass
        else:
            self.pre, self.post = self.pre_transform(pre), self.post_transform(post)
            return self.pre, self.post


    def pre_transform(self,x):

        x = torch.from_numpy(x).permute(2,0,1).float()
        
        x = normalize(x,self.pre_meanstd['mean'],self.pre_meanstd['std'],inplace=True)

        return x

    def post_transform(self,x):

        x  = torch.from_numpy(x).permute(2,0,1).float()
        
        x = normalize(x,self.post_meanstd['mean'],self.post_meanstd['std'],inplace=True)

        return x
    
    
    def __len__(self):
        return len(self.pre)
    
dataset = CustomDataset(pre_pathlist,post_pathlist)

train_loader = DataLoader(dataset,shuffle=True,batch_size=200)

NameError: name 'Dataset' is not defined

In [ ]:
#Position Embeddings implementation


In [12]:
# Model

# Pre trained VIT encoder

class VITencoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = tvmodels.vit_b_16(weights=True)

        self.encoder.heads = nn.Identity()


    def forward (self,x):

        x = self.encoder(x)

        return x
    
#Linear lightweight FCN layer decoder

class LightDecoderM(nn.Module):

    def __init__(self, ):
        super().__init__()

        self.decoder = nn.Sequential(
            nn.Conv2d(
                in_channels=768,
                out_channels=self.encoder_stride ** 2 * 3, kernel_size=1),
            nn.PixelShuffle(self.encoder_stride),
        )

        self.in_chans = self.encoder.in_chans
        self.patch_size = self.encoder.patch_size

    def forward(self,x):

        x = self.decoder(x)

        return x


In [ ]:
# Build Sharedweight Encoder

In [ ]:
# Training Pipeline
